## 6.3 Simple RNN前向传播 - 损失函数

#### 1、这一节我们要解决什么问题 🎯

前面我们已经学习了：

- RNN 的基本结构
- 单时间步的前向传播
- 整个序列的前向传播
- 序列与 batch 下的张量形状
- RNN 的输入输出模式：$1:1$、$1:N$、$N:1$、$N:N$

但是到这里，还缺少神经网络训练中非常关键的一环：

模型算出了输出之后，如何判断它算得好不好？

这就需要引入：

- 损失函数（Loss Function）

##### 1️⃣ RNN 的损失函数是什么？

##### 2️⃣ RNN 会不会有“专门独有”的损失函数？

##### 3️⃣ 为什么 RNN 的难点不在损失函数本身，而在“loss 落在哪些时间步”？

##### 4️⃣ 在不同输入输出模式下，loss 应该怎么计算？

##### 5️⃣ $N:1$ 和 $N:N$ 下的 loss 有什么区别？

RNN 的损失函数本身通常并不新，很多仍然是我们之前学过的交叉熵、MSE 等；  
真正新的地方在于，RNN 的输出发生在时间维度上，因此 loss 也会和时间维度发生关系。

#### 2、我们之前已经学过哪些损失函数 📦

在 MLP 和 CNN 中，我们已经接触过一些常见损失函数，例如：

- 交叉熵损失（Cross Entropy Loss）
- 二分类交叉熵（Binary Cross Entropy）
- 均方误差（MSE/L2）
- 平均绝对值误差（MAE/L1）
- Smooth L1 Loss（L1 + L2）

RNN 在很多任务中，使用的仍然是这些损失函数。

#### 3、RNN 的损失函数“新”在哪里 🧠

##### 3.1 新的不一定是损失函数本身

RNN 和 MLP、CNN 相比，一个最大的不同是：

- MLP / CNN 往往输出一次结果
- RNN 可能会在多个时间步产生输出

所以问题就来了：

loss 应该在哪个时间步上计算？

这才是 RNN 损失函数学习中的关键。

##### 3.2 RNN 的输出可能出现在不同位置

例如：

###### （1）只在最后一步输出

这对应：

- 文本分类
- 情感分析
- 序列整体分类

此时通常只会有一个最终输出。

###### （2）每一个时间步都输出

这对应：

- 词性标注
- 命名实体识别
- 每一步时间序列预测

此时会有一串输出。

###### （3）先编码，再逐步解码输出

这对应：

- 机器翻译
- 序列生成
- 文本生成

此时也会有多个时间步输出，但输出发生在解码阶段。

##### 3.3 所以 RNN 的核心问题是：loss 落点不同

你可以这样理解：

RNN 的损失函数本体常常和以前一样，但 loss 的计算位置和聚合方式变了。

也就是说，RNN 的重点不在“发明了新 loss”，而在：

- loss 对最后一步算，还是对所有时间步都算
- 多个时间步的 loss 如何合并
- 哪些时间步参与训练，哪些时间步忽略

#### 4、先从最简单的情况开始：单输出 loss 🔹

这是最容易理解的一类，也就是：

- 模型读入一个序列
- 最后只输出一个结果

这通常对应：

- $N:1$

例如：

- 文本情感分类
- 垃圾邮件分类
- 语音整体分类
- 时间序列整体预测

##### 4.1 典型形式

假设输入序列为：

- $x_1, x_2, x_3, \dots, x_t$

RNN 逐步计算隐藏状态：

- $h_1, h_2, h_3, \dots, h_t$

最后只取最终输出：

- $y_t$

然后把这个输出和真实标签 $\hat{y}$ 进行比较，计算一个 loss。

也就是说：

前面所有时间步都在“读”和“记”，最后一步才真正拿来和标签比较。

##### 4.2 如果是分类任务

例如情感分类：

- 输入一句评论
- 输出 positive / negative

那么最常用的损失函数仍然是：

- 交叉熵损失

如果模型最后输出类别 logits：

- $y_t.shape = (1, num\_classes)$

真实标签是一个类别 id：

- $\hat{y}$

那么 loss 可以写成：

$L = CrossEntropy(y_t, \hat{y})$

这和我们之前在 MLP / CNN 分类中学的是一样的。

##### 4.3 如果是回归任务

例如：

- 输入前 $T$ 天温度
- 预测第 $T+1$ 天数值

那么最后一步输出通常是一个实数或一个向量，  
此时最常见的损失函数仍然是：

- MSE Loss

即：

$L = MSE(y_t, \hat{y})$

##### 4.4 这一类任务的核心特点

这类任务中：

- 序列前面的时间步不直接计算 loss
- 但它们会通过影响最终隐藏状态 $h_t$，间接影响最终 loss

所以你可以这样理解：

虽然 loss 只在最后一步计算，但整个序列都在为最后一步服务。

#### 5、再看多输出 loss：每一步都参与计算 🔸

这类任务中，RNN 会在每一个时间步都输出一个结果。

这通常对应：

- 同步型 $N:N$

例如：

- 词性标注
- 命名实体识别
- 每一步时间序列预测
- 每个语音帧预测一个标签

##### 5.1 典型形式

输入序列：

- $x_1, x_2, x_3, \dots, x_t$

模型输出：

- $y_1, y_2, y_3, \dots, y_t$

真实标签也对应每一步：

- $\hat{y}_1, \hat{y}_2, \hat{y}_3, \dots, \hat{y}_t$

这时，每个时间步都可以单独算一个 loss：

- $L_1$
- $L_2$
- $L_3$
- $\dots$
- $L_t$

然后再把这些 loss 合并起来。

##### 5.2 最常见的合并方式：求和或求平均

例如：

$L = \sum_{t=1}^{T} L_t$

或者：

$L = \frac{1}{T} \sum_{t=1}^{T} L_t$

这表示：

每一个时间步都要为最终训练目标负责。

##### 5.3 如果每一步都是分类

例如词性标注任务中：

- 第 $t$ 个词输出一个类别分布
- 再和第 $t$ 个真实标签比较

那么每一步可以用交叉熵：

$L_t = CrossEntropy(y_t, \hat{y}_t)$

整条序列 loss 就是：

$L = \sum_{t=1}^{T} CrossEntropy(y_t, \hat{y}_t)$

或者取平均。

##### 5.4 如果每一步都是回归

例如：

- 每个时间步预测一个连续值

那么每一步可以用 MSE：

$L_t = MSE(y_t, \hat{y}_t)$

整条序列 loss 就可以写成：

$L = \sum_{t=1}^{T} MSE(y_t, \hat{y}_t)$

或者取平均。

##### 5.5 这一类任务的核心特点

这类任务中：

- loss 不只来自最后一步
- 而是来自整条序列多个时间步

所以训练信号更“分散”在整个时间维度上。

你可以理解为：

模型不是最后才被评分，而是在每一步都被评分。


#### 6、再看生成类任务中的 loss 📖

这类任务中，RNN 常用于：

- 文本生成
- 序列生成
- 解码器输出序列
- 机器翻译（早期 RNN encoder-decoder）

##### 6.1 典型形式

例如生成一句文本：

- 输入起始标记 `<start>`
- 模型输出第一个词
- 再输出第二个词
- 再输出第三个词
- $\dots$
- 直到结束标记 `<end>`

这时，模型其实是在很多时间步上不断生成输出。

每一步都有一个真实目标词，所以每一步都可以计算一个损失。

##### 6.2 本质上仍然是“每步一个 loss”

假设目标序列长度为 $T$，那么：

- 第 1 步输出对应真实词 $\hat{y}_1$
- 第 2 步输出对应真实词 $\hat{y}_2$
- $\dots$
- 第 $T$ 步输出对应真实词 $\hat{y}_t$

于是总 loss 仍然可以写成：

$L = \sum_{t=1}^{T} L_t$

如果是词预测任务，那么每一步通常仍然是：

- 交叉熵损失

所以本质上它还是多时间步 loss。

##### 6.3 为什么它和同步型 $N:N$ 很像

因为从 loss 的角度看，它们都属于：

多个时间步输出，多个时间步参与损失计算。

区别只是：

- 同步型 $N:N$：输入和输出时间步通常对齐
- 生成型任务：输出是逐步生成的，不一定和输入步同步对应

但对于 loss 来说，本质都可以看成：

- 每一步各算一个 loss
- 最后再聚合

#### 7、batch 下的损失函数怎么理解 🚚

前面我们讲的是单条序列。  
真实训练中，通常会加上 batch。

##### 7.1 先看 $N:1$ 情况

假设：

- batch 中有 $B$ 条序列
- 每条序列最终输出一个结果

那么：

- 预测输出 shape 可能是：$(B, num\_classes)$
- 标签 shape 可能是：$(B,)$

这时 loss 函数会同时比较 batch 中所有样本的输出和标签。

例如分类任务中：

$Loss = CrossEntropy(Y, \hat{Y})$

这里本质上就是：

对 batch 中每一条序列的最终输出分别计算 loss，再做求和或平均。

##### 7.2 再看 $N:N$ 情况

假设：

- batch 中有 $B$ 条序列
- 每条序列长度为 $T$
- 每一步都输出

那么输出张量可能是：

- $(B, T, num\_classes)$

标签张量可能是：

- $(B, T)$

此时本质上仍然是：

对 batch 中每条序列、每个时间步分别计算 loss，再在 batch 和时间维度上聚合。

也就是说：

- 先有时间维度上的多个 loss
- 再有 batch 维度上的多个样本
- 最后统一求和或平均

##### 7.3 所以 batch 不是改变损失函数本质

batch 的加入，不会改变 loss 的本质类型，它只是让：

- 更多样本并行参与训练

所以你可以这样理解：

batch 只是把“单条序列的 loss 计算”扩展成“多条序列并行的 loss 计算”。